# Анализ влияния погодных условий на ДТП

## Цель и задачи проекта
**Цель**: сбор погодных данных и данных о ДТП для выбранных городов в Российской Федерации и проведение анализа для 
выявления взаимосвязей между ними.

**Задачи данного этапа**:
- Загрузка ключевых данных: 
-- Список городов РФ;
-- Координаты городов;
-- Коды ОКАТО регионов РФ;
-- Данные о погоде;
-- Карточки ДТП;
- Проведение очистки и предобработки данных;
- Загрузка данных в базу данных.

## Содержание:
1. [Парсинг списка городов со страницы Википедии](#soderjanie_1)
2. [Создание соединения к базе данных](#soderjanie_2)
3. [Предобработка и обогащение данных (геокодирование (добавление координат)) о городах РФ и их выгрузка в таблицу базы данных](#soderjanie_3)
4. [Парсинг данных о погоде через API](#soderjanie_4)
5. [Предобработка данных о погоде в выбранных городах и выгрузка в таблицу базы данных](#soderjanie_5)
6. [Получение ОКАТО-кодов регионов РФ](#soderjanie_6)
7. [Предобработка данных о кодах ОКАТО и их выгрузка в таблицу базы данных](#soderjanie_7)
8. [Парсинг данных о ДТП через API](#soderjanie_8)
9. [Предобработка данных о ДТП и их выгрузка в таблицу базы данных](#soderjanie_9)
10. [Выводы](#soderjanie_10)


## 1. Парсинг списка городов со страницы Википедии <a id='soderjanie_1'></a>

In [1]:
#Устанавливаем библиотеки

!pip install geopy
!pip install requests_cache

In [2]:
#Импортируем библиотеки

import requests
import requests_cache
import pandas as pd
import numpy as np
import logging
from bs4 import BeautifulSoup
from tqdm import tqdm
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from typing import List, Optional
import json
import time
from datetime import datetime
import sqlalchemy

In [3]:
#Сохраняем список выбранных городов

cities_selected=['Казань','Новосибирск']

In [4]:
#Функция для логирования

def setup_logging() -> logging.Logger:

    '''Настройка логирования.'''

    log_filename = f'common.log'  # Название файла логов

    logger = logging.getLogger('dtp_weather') # Название логера
    logger.setLevel(logging.INFO)

    # Очистка существующих обработчиков
    if logger.handlers:
        logger.handlers.clear()

    # Обработчик для файла
    file_handler = logging.FileHandler(log_filename, encoding='utf-8')
    # Формат вывода сообщений логов в файл
    file_formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    file_handler.setFormatter(file_formatter)
    logger.addHandler(file_handler)

    # Обработчик для консоли
    console_handler = logging.StreamHandler()
    # Формат вывода логов в консоль (сокращенный)
    console_formatter = logging.Formatter('%(levelname)s: %(message)s')
    console_handler.setFormatter(console_formatter)
    logger.addHandler(console_handler)

    return logger

In [5]:
#Функция для получения HTML-кода страницы по заданному URL

def get_html(url: str,
             headers: str) -> str:

    '''Получает HTML-код страницы по заданному URL.'''

    try:
        logger.info(f'Попытка получить HTML-код страницы: {url}')
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()  # Проверка на ошибки HTTP
        logger.info('HTML-код успешно получен.')
        return response.text
    except requests.exceptions.RequestException as e:
        logger.error(f'Ошибка при получении HTML-кода: {e}')
        raise

In [6]:
#Функция для получения таблицы с административными центрами из HTML-кода

def parse_table(html: str) -> list:

    '''Парсит таблицу с административными центрами из HTML-кода.'''

    try:
        logger.info('Парсинг HTML-кода для извлечения таблицы.')
        soup = BeautifulSoup(html, 'html.parser')
        table = soup.find('table')

        if not table:
            logger.error('Таблица не найдена на странице.')
            raise ValueError('Таблица не найдена.')

        rows = table.find_all('tr')
        if not rows:
            logger.error('Строки таблицы не найдены.')
            raise ValueError('Строки таблицы не найдены.')

        data = []

        for row in rows:
            cols = row.find_all('td')
            cols = [col.text.strip() for col in cols]
            data.append(cols)

        logger.info(f'Успешно извлечено {len(data)} строк из таблицы.')
        return data
    except Exception as e:
        logger.error(f'Ошибка при парсинге таблицы: {e}')
        raise

In [7]:
#Функция для сохрания данных в датафрейм и CSV-файл

def save_to_csv_DataFrame(data: list, filename: str) -> None:

    '''Сохраняет данные в датафрейм и CSV-файл'''
    try:
        logger.info(f'Сохранение данных в датафрейм и файл: {filename}')
        columns = [
            'num',
            'coat_of_arms',
            'city_name',
            'region',
            'federal',
            'population',
            'foundation_or_first_mention',
            'year_of_city_status',
            'former_city_name'
        ]

        df = pd.DataFrame(data[1:], columns=columns)
        df.to_csv(filename, index=False, encoding='utf-8')
        logger.info(f'Данные успешно сохранены в датафрейм и файл: {filename}')
        return df
    except Exception as e:
        logger.error(f'Ошибка при сохранении данных в датафрейм и CSV: {e}')
        raise

In [8]:
#Выполняем функции для получения списка городов и сохранения их в датафрейм и в файл

if __name__ =='__main__':  # этот блок кода работает, когда файл запущен напрямую
    # Сохраняем строкy подключения к русской Википедии
    url_cities = 'https://ru.wikipedia.org/wiki/%D0%A1%D0%BF%D0%B8%D1%81%D0%BE%D0%BA_%D0%B3%D0%BE%D1%80%D0%BE%D0%B4%D0%BE%D0%B2_%D0%A0%D0%BE%D1%81%D1%81%D0%B8%D0%B8'
    
    # Указываем заголовок
    user_agent='Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36'
    headers = {'User-Agent': user_agent}
    
    # Указываем имя файла
    output_filename = 'cities_rf.csv'
    
    # Создаем логер
    logger = setup_logging()
     
    #Выполняем функции
    try:
        html_city = get_html(url_cities, headers)
        data_city = parse_table(html_city)
        df_city = save_to_csv_DataFrame(data_city, output_filename)
        
    except Exception as e:
        logger.error(f"Ошибка в процессе выполнения скрипта: {e}")

INFO: Попытка получить HTML-код страницы: https://ru.wikipedia.org/wiki/%D0%A1%D0%BF%D0%B8%D1%81%D0%BE%D0%BA_%D0%B3%D0%BE%D1%80%D0%BE%D0%B4%D0%BE%D0%B2_%D0%A0%D0%BE%D1%81%D1%81%D0%B8%D0%B8
INFO: HTML-код успешно получен.
INFO: Парсинг HTML-кода для извлечения таблицы.
INFO: Успешно извлечено 1126 строк из таблицы.
INFO: Сохранение данных в датафрейм и файл: cities_rf.csv
INFO: Данные успешно сохранены в датафрейм и файл: cities_rf.csv


_____

## 2. Создание соединения к базе данных <a id='soderjanie_2'></a>

In [9]:
# Параметры базы данных

USER ='postgres.pkhitenuyhdiohvrxklz'
PASSWORD = 'Stop_Start_7'
HOST = 'aws-1-eu-central-2.pooler.supabase.com'
PORT = '6543'
DBNAME = 'postgres'

In [10]:
# Сформируем строку подключения к SQLAlchemy

DATABASE_URL = f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DBNAME}?sslmode=require"

In [11]:
# Создаем соединение

engine = sqlalchemy.create_engine(DATABASE_URL)
logger.info("Создаем подключение к базе данных")

INFO: Создаем подключение к базе данных


In [13]:
# Tест соединения

try:
    with engine.connect() as connection:
        logger.info("Connection successful!")
except Exception as e:
    logger.error(f"Failed to connect: {e}")

INFO: Connection successful!


___

## 3. Предобработка и обогащение данных (геокодирование (добавление координат)) о городах РФ и их выгрузка в таблицу базы данных <a id='soderjanie_3'></a>

In [ ]:
#Выгружаем полученные данные о городах в буферную таблицу базы данных

logger.info("Выгружаем полученные данные о городах в буферную таблицу базы данных")
df_city.to_sql('bufer_cities', con=engine, if_exists='replace', index=False)
logger.info("Данные о городах успешно выгружены в буферную таблицу базы данных")

In [14]:
# Выводим информацию о датафрейме

df_city.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1125 entries, 0 to 1124
Data columns (total 9 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   num                          1125 non-null   object
 1   coat_of_arms                 1125 non-null   object
 2   city_name                    1125 non-null   object
 3   region                       1125 non-null   object
 4   federal                      1125 non-null   object
 5   population                   1125 non-null   object
 6   foundation_or_first_mention  1125 non-null   object
 7   year_of_city_status          1125 non-null   object
 8   former_city_name             1125 non-null   object
dtypes: object(9)
memory usage: 79.2+ KB


In [15]:
# Выводим первые 5 строк датафрейма с городами

df_city.head()

,num,coat_of_arms,city_name,region,federal,population,foundation_or_first_mention,year_of_city_status,former_city_name
0,1,,Абаза,Хакасия,Сибирский,12 272,1867,1966,"Абаканский Завод, Абаканско-Заводское"
1,2,,Абакан,Хакасия,Сибирский,184 769,1734,1931,Усть-Абаканское (до 1931)
2,3,,Абдулино,Оренбургская область,Приволжский,17 274,1795,1923,
3,4,,Абинск,Краснодарский край,Южный,39 511,1863,1963,Абинское (до 1863);Абинская (до 1962)
4,5,,Агидель,Башкортостан,Приволжский,14 219,1980,1991,


In [16]:
# Оставляем необходимые для работы колонки

df_city=df_city[['city_name', 'region', 'federal', 'population']]

In [17]:
# Проверяем уникальные значения федеральных округов

df_city['federal'].unique()

array(['Сибирский', 'Приволжский', 'Южный', 'Северо-Кавказский',
       'Уральский', 'Дальневосточный', 'Центральный', 'Северо-Западный'],
      dtype=object)

In [18]:
# Проверяем уникальные значения регионов

df_city['region'].unique()

array(['Хакасия', 'Оренбургская область', 'Краснодарский край',
       'Башкортостан', 'Татарстан', 'Адыгея', 'Ростовская область',
       'Тыва', 'Северная Осетия', 'Свердловская область', 'Чувашия',
       'Якутия', 'Алтайский край', 'Владимирская область',
       'Пермский край', 'Сахалинская область', 'Белгородская область',
       'Тульская область', 'Иркутская область', 'Крым',
       'Хабаровский край', 'Чукотский АО', 'Тверская область',
       'Кемеровская область', 'Мурманская область', 'Московская область',
       'Чечня', 'Мордовия', 'Нижегородская область',
       'Саратовская область', 'Приморский край', 'Красноярский край',
       'Архангельская область', 'Томская область', 'Астраханская область',
       'Челябинская область', 'Вологодская область', 'Бурятия',
       'Калининградская область', 'Кабардино-Балкария',
       'Калужская область', 'Забайкальский край', 'Новосибирская область',
       'Ульяновская область', 'Кировская область', 'Пензенская область',
       'Ам

In [19]:
# Преобразуем данные о названии региона

df_city['region'] = df_city['region'].str.replace('Еврейская АО', 'Еврейская Автономная область')
df_city['region'] = df_city['region'].str.replace(' АО', ' Автономный округ')

In [20]:
# Преобразуем и очистим данные о названии городов

df_city['city_name'] = df_city['city_name'].str.replace('не призн.', '').str.strip()

/tmp/ipykernel_338/2485007527.py:3: FutureWarning: The default value of regex will change from True to False in a future version.
  df_city['city_name'] = df_city['city_name'].str.replace('не призн.', '').str.strip()


In [21]:
# Преобразуем и очистим данные о населении городов

df_city['population'] = df_city['population'].str.replace(' ', '').str.split('[').str[0].astype('int')

In [22]:
# Добавляем пустые колонки для координат

df_city['lat'], df_city['lon'] = None, None

In [23]:
#Функция для получения координат по названию города через геокодер Nominatim

def coordinates(user_agent: str):

    '''
    Функция для получения координат по названию города через геокодер Nominatim.

    '''
    try:
        logger.info(f'Заполнение данных о широте и долготе в соответствующих полях датафрейма')
        geolocator = Nominatim(user_agent=user_agent)
        geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)
        for index, row in df_city.iterrows():
            state=row['region']
            city=row['city_name']
            location = geocode(f'{state}, {city}')
            if location:
                df_city.loc[index, 'lon']=location.longitude
                df_city.loc[index, 'lat']=location.latitude
            else:
                df_city.loc[index, 'lon']='Нет данных'
                df_city.loc[index, 'lat']='Нет данных'
                
        logger.info(f'Данные о широте и долготе успешно заполнены')
        return df_city
    except Exception as e:
        logger.error(f'Ошибка при заполнении данных о широте и долготе: {e}')
        raise

In [24]:
#Выполняем функцию для сохранения обогащенного датафрейма с координатами городов

user_agent = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36'
df_city = coordinates(user_agent)

INFO: Заполнение данных о широте и долготе в соответствующих полях датафрейма
INFO: Данные о широте и долготе успешно заполнены


In [25]:
# Проверяем наличие незаполненных данных

df_city[(df_city['lon']=='Нет данных')| (df_city['lat']=='Нет данных')] 

,city_name,region,federal,population,lat,lon


In [26]:
#Фильтруем датафрейм от незаполненных данных

df_city=df_city[(df_city['lon']!='Нет данных')& (df_city['lat']!='Нет данных')] 

In [27]:
# Выводим информацию о датафрейме

df_city.info() 

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1125 entries, 0 to 1124
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   city_name   1125 non-null   object
 1   region      1125 non-null   object
 2   federal     1125 non-null   object
 3   population  1125 non-null   int64 
 4   lat         1125 non-null   object
 5   lon         1125 non-null   object
dtypes: int64(1), object(5)
memory usage: 61.5+ KB


In [28]:
# Изменяем тип данных полей с координатами на float

df_city['lon'] = df_city['lon'].astype('float')
df_city['lat'] = df_city['lat'].astype('float')

In [29]:
# Применяем метод nunique() к столбцу city_name

df_city['city_name'].nunique()

1105

In [30]:
#Сохраняем и выводим значения-дубликаты по полю с названием городов

dupl_cities=df_city.loc[(df_city['city_name'].duplicated())]['city_name']
dupl_cities

86           Белогорск
99         Берёзовский
109       Благовещенск
234           Гурьевск
291       Железногорск
313           Заречный
412              Киров
416            Кировск
465      Красноармейск
471     Краснознаменск
476     Краснослободск
593             Мирный
597         Михайловск
658           Никольск
705             Озёрск
783           Приморск
801           Радужный
886            Советск
887            Советск
1026            Фокино
Name: city_name, dtype: object

In [31]:
#Выводим строки датафрейма по дублирующимся городам для проверки заполнения широты и долготиы

df_city[df_city['city_name'].isin(dupl_cities)]

,city_name,region,federal,population,lat,lon
85,Белогорск,Амурская область,Дальневосточный,61440,50.921326,128.472870
86,Белогорск,Крым,Южный,17445,45.058439,34.594959
98,Берёзовский,Кемеровская область,Сибирский,44932,55.669695,86.275154
99,Берёзовский,Свердловская область,Уральский,59698,56.909787,60.812025
108,Благовещенск,Башкортостан,Приволжский,35481,55.046236,55.961169
109,Благовещенск,Амурская область,Дальневосточный,241437,50.260042,127.533738
233,Гурьевск,Калининградская область,Северо-Западный,26760,54.774557,20.603704
234,Гурьевск,Кемеровская область,Сибирский,22134,54.284954,85.931091
290,Железногорск,Красноярский край,Сибирский,82723,56.250938,93.532860
291,Железногорск,Курская область,Центральный,97038,52.338946,35.351175


In [32]:
# Выводим первые 10 строк датафрейма, отсортированного по населению, на экран

df_city.sort_values(by='population', ascending=False).head(10)

,city_name,region,federal,population,lat,lon
607,Москва,Москва,Центральный,13010112,55.750541,37.617478
831,Санкт-Петербург,Санкт-Петербург,Северо-Западный,5601911,59.938732,30.316229
676,Новосибирск,Новосибирская область,Сибирский,1633595,55.028831,82.922689
277,Екатеринбург,Свердловская область,Уральский,1544376,56.838207,60.600789
354,Казань,Татарстан,Приволжский,1308660,55.794649,49.111502
651,Нижний Новгород,Нижегородская область,Приволжский,1226076,56.326482,44.005139
1053,Челябинск,Челябинская область,Уральский,1189525,55.159841,61.402555
480,Красноярск,Красноярский край,Сибирский,1187771,56.009117,92.872586
830,Самара,Самарская область,Приволжский,1173299,53.195625,50.101493
1019,Уфа,Башкортостан,Приволжский,1144809,54.726141,55.947499


In [ ]:
#Выгружаем обработанные данные о городах в таблицу базы данных

logger.info("Выгружаем полученные данные о городах в таблицу базы данных")
df_city.to_sql('cities', con=engine, if_exists='replace', index=False)
logger.info("Данные о городах успешно выгружены в таблицу базы данных")

___

## 4. Парсинг данных о погоде через API<a id='soderjanie_4'></a>

In [33]:
# Устанавливаем кэш с бесконечным сроком жизни (-1): создаст файл 'api_cache.sqlite'

requests_cache.install_cache('api_cache', backend='sqlite', expire_after=-1)

In [34]:
# Функция для получения данных о погоде через API сайта open-meteo.com

def weather (weather_cities: List[str],
             headers: dict,
             start_date: str,
             end_date: str) -> pd.DataFrame:

    '''
    Функция для получения данных о погоде через API сайта open-meteo.com.
    Args:
        weather_cities: список городов
        headers: заголовок
        start_date: дата начала периода в формате ISO8601: 0000-00-00
        end_date: дата окончания периода в формате ISO8601: 0000-00-00
    '''
    # Список для хранения данных о показателях
    all_data_weather = []
    
    # Сохраняем адрес, по которому обращаемся для получения данных
    url_weather = 'https://archive-api.open-meteo.com/v1/archive'
    logger.info(f'Парсинг данных о погоде из API сайта open-meteo.com')
     
    for weather_city in weather_cities:
        try:
            params_weather = {
            'latitude': df_city[df_city['city_name']==weather_city]['lat'],
            'longitude': df_city[df_city['city_name']==weather_city]['lon'],
            'start_date': start_date,
            'end_date': end_date,
            'hourly': [
                'temperature_2m', 'relative_humidity_2m', 'dew_point_2m', 'apparent_temperature',
                'pressure_msl', 'surface_pressure', 'precipitation', 'rain', 'snowfall', 'cloud_cover', 
                'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_high', 'shortwave_radiation', 
                'direct_radiation', 'direct_normal_irradiance', 'diffuse_radiation', 
                'global_tilted_irradiance', 'sunshine_duration', 'wind_speed_10m', 'wind_speed_100m', 
                'wind_direction_10m', 'wind_direction_100m', 'wind_gusts_10m', 'et0_fao_evapotranspiration', 
                'weather_code', 'snow_depth', 'vapour_pressure_deficit', 'soil_temperature_0_to_7cm', 
                'soil_temperature_7_to_28cm', 'soil_temperature_28_to_100cm', 
                'soil_temperature_100_to_255cm','soil_moisture_0_to_7cm', 'soil_moisture_7_to_28cm', 
                'soil_moisture_28_to_100cm', 'soil_moisture_100_to_255cm'
            ],
            'daily': [
                'weather_code', 'temperature_2m_max', 'temperature_2m_min', 'apparent_temperature_max', 
                'apparent_temperature_min', 'precipitation_sum', 'rain_sum', 'snowfall_sum', 
                'precipitation_hours', 'sunrise', 'sunset', 'sunshine_duration', 'daylight_duration', 
                'wind_speed_10m_max', 'wind_gusts_10m_max', 'wind_direction_10m_dominant',
                'shortwave_radiation_sum', 'et0_fao_evapotranspiration'
            ],
            'timezone': 'auto'}
            # Отправляем GET запрос к API 
            response_weather = requests.get(url_weather, params=params_weather, headers=headers, timeout=10)
            
            response_weather.raise_for_status()  # Вызовет ошибку, если статус не 200-299
            logger.info(f'Запрос данных о погоде в городе {weather_city} успешно выполнен. {response_weather.from_cache=}')
             # Обрабатываем возможные ошибки при работе с API
        except requests.exceptions.RequestException as e:
            logger.error(f"Ошибка при запросе к API: {e}")
            raise
        except (KeyError, IndexError, ValueError) as e:
            logger.error(f"Ошибка при обработке данных: {e}")
            raise
        # Десериализуем полученный JSON-ответ. 
        # Парсим ответ от API 
        data_weather = response_weather.json()
        # Добавляем данные в общий список
        all_data_weather.append(data_weather)
            
    if all_data_weather:
        logger.info(f'Парсинг данных о погоде из API сайта open-meteo.com успешно завершен')
            
    return all_data_weather
    

In [35]:
# Выполняем функцию для сохранения данных о погоде

# Указываем заголовок
headers = {'User-Agent': user_agent}

all_data_weather=weather(cities_selected, headers, '2016-01-01', '2025-12-31')

INFO: Парсинг данных о погоде из API сайта open-meteo.com
INFO: Запрос данных о погоде в городе Казань успешно выполнен. response_weather.from_cache=True
INFO: Запрос данных о погоде в городе Новосибирск успешно выполнен. response_weather.from_cache=True
INFO: Парсинг данных о погоде из API сайта open-meteo.com успешно завершен


___

## 5. Предобработка данных о погоде в выбранных городах и выгрузка в таблицу базы данных <a id='soderjanie_5'></a>

In [36]:
#Сохраняем полученные данные о погоде в датафрейм

df_weather = pd.json_normalize(all_data_weather, sep='_')

In [37]:
# Выводим первые 5 строк датафрейма c погодой

df_weather.head()

,latitude,longitude,generationtime_ms,utc_offset_seconds,timezone,timezone_abbreviation,elevation,hourly_units_time,hourly_units_temperature_2m,hourly_units_relative_humidity_2m,...,daily_precipitation_hours,daily_sunrise,daily_sunset,daily_sunshine_duration,daily_daylight_duration,daily_wind_speed_10m_max,daily_wind_gusts_10m_max,daily_wind_direction_10m_dominant,daily_shortwave_radiation_sum,daily_et0_fao_evapotranspiration
0,55.782074,49.124237,5248.989224,10800,Europe/Moscow,GMT+3,82.0,iso8601,°C,%,...,"[0.0, 0.0, 14.0, 0.0, 4.0, 0.0, 14.0, 5.0, 22....","[2016-01-01T08:13, 2016-01-02T08:12, 2016-01-0...","[2016-01-01T15:20, 2016-01-02T15:21, 2016-01-0...","[19241.01, 1478.63, 0.0, 3582.74, 1657.6, 1779...","[25612.66, 25700.26, 25795.33, 25897.61, 26006...","[16.6, 13.5, 24.1, 26.5, 23.8, 16.3, 18.1, 16....","[29.9, 21.2, 46.1, 45.4, 42.5, 27.4, 33.5, 27....","[50, 35, 65, 128, 143, 113, 150, 151, 127, 139...","[2.47, 1.69, 1.12, 1.86, 1.85, 2.27, 1.35, 1.6...","[0.14, 0.07, 0.18, 0.19, 0.16, 0.12, 0.14, 0.1..."
1,55.079086,82.994020,202.665448,25200,Asia/Novosibirsk,GMT+7,149.0,iso8601,°C,%,...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[2016-01-01T09:53, 2016-01-02T09:53, 2016-01-0...","[2016-01-01T17:08, 2016-01-02T17:10, 2016-01-0...","[17764.07, 19708.06, 19726.37, 20212.0, 19626....","[26133.68, 26216.59, 26306.72, 26403.85, 26507...","[10.7, 14.8, 15.1, 14.1, 20.0, 21.7, 11.4, 11....","[16.6, 24.8, 26.3, 24.5, 37.1, 38.5, 19.4, 20....","[337, 34, 39, 34, 51, 88, 104, 169, 328, 50, 8...","[2.45, 2.83, 2.82, 2.74, 2.86, 2.94, 2.23, 2.0...","[0.06, 0.08, 0.1, 0.1, 0.16, 0.24, 0.12, 0.12,..."


In [ ]:
#Выгружаем полученные данные о погоде в буферную таблицу базы данных

logger.info("Выгружаем полученные данные о погоде в буферную таблицу базы данных")
for i, col in enumerate(df_weather.columns):
    if i==0:
        df_col = df_weather[col]
        df_col.to_sql('bufer_weather', con=engine, if_exists='replace', index=False, chunksize=500, method='multi')
    else:
        df_col= df_weather[[df_weather.columns[0],col]]

        df_col.to_sql('temp_table', con=engine, if_exists='replace', index=False, chunksize=500, method='multi')
        with engine.connect() as conn:
            conn.execute(f"ALTER TABLE bufer_weather ADD COLUMN {col} text")

            conn.execute(f"UPDATE bufer_weather SET {col} =(SELECT {col} FROM temp_table  WHERE bufer_weather.{df_weather.columns[0]} = temp_table.{df_weather.columns[0]})")
            conn.execute("DROP TABLE temp_table")
             
logger.info("Данные о погоде успешно выгружены в буферную таблицу базы данных")   

In [38]:
# Сохраняем данные почасовые данные о погоде в датафрейм

df_all_data_weather_hourly = pd.DataFrame()
for item in ['time','temperature_2m', 'relative_humidity_2m', 'dew_point_2m', 'apparent_temperature', 
                    'pressure_msl', 'surface_pressure', 'precipitation', 'rain', 'snowfall', 'cloud_cover', 
                    'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_high', 'shortwave_radiation', 
                    'direct_radiation', 'direct_normal_irradiance', 'diffuse_radiation', 
                    'global_tilted_irradiance', 'sunshine_duration', 'wind_speed_10m', 'wind_speed_100m', 
                    'wind_direction_10m', 'wind_direction_100m', 'wind_gusts_10m', 'et0_fao_evapotranspiration', 
                    'weather_code', 'snow_depth', 'vapour_pressure_deficit', 'soil_temperature_0_to_7cm', 
                    'soil_temperature_7_to_28cm', 'soil_temperature_28_to_100cm', 
                    'soil_temperature_100_to_255cm','soil_moisture_0_to_7cm', 'soil_moisture_7_to_28cm', 
                    'soil_moisture_28_to_100cm', 'soil_moisture_100_to_255cm']:
    if not df_all_data_weather_hourly.empty:
        df_all_data_weather_hourly[item]=pd.json_normalize(all_data_weather,
        meta=['latitude', 'longitude','generationtime_ms', 'utc_offset_seconds',
              'timezone','timezone_abbreviation','elevation'],
        record_path=['hourly', item],
        record_prefix='item_')['item_0']
    else:
        df_all_data_weather_hourly=pd.json_normalize(all_data_weather,
                          meta=['latitude', 'longitude','generationtime_ms', 'utc_offset_seconds',
                                'timezone','timezone_abbreviation','elevation'],
                          record_path=['hourly', item],
                          record_prefix='item_'
                          )
        df_all_data_weather_hourly.rename(columns={'item_0': item}, inplace=True)

In [39]:
# Сохраняем данные дневные о погоде в датафрейм

df_all_data_weather_daily = pd.DataFrame()
for item in ['time','weather_code', 'temperature_2m_max', 'temperature_2m_min', 'apparent_temperature_max', 
                    'apparent_temperature_min', 'precipitation_sum', 'rain_sum', 'snowfall_sum', 
                    'precipitation_hours', 'sunrise', 'sunset', 'sunshine_duration', 'daylight_duration', 
                    'wind_speed_10m_max', 'wind_gusts_10m_max', 'wind_direction_10m_dominant',
                    'shortwave_radiation_sum', 'et0_fao_evapotranspiration']:
    if not df_all_data_weather_daily.empty:
        df_all_data_weather_daily[item]=pd.json_normalize(all_data_weather,
        meta=['latitude', 'longitude','generationtime_ms', 'utc_offset_seconds',
              'timezone','timezone_abbreviation','elevation'],
        record_path=['daily', item],
        record_prefix='item_')['item_0']
    else:
        df_all_data_weather_daily=pd.json_normalize(all_data_weather,
                          meta=['latitude', 'longitude','generationtime_ms', 'utc_offset_seconds',
                                'timezone','timezone_abbreviation','elevation'],
                          record_path=['daily', item],
                          record_prefix='item_'
                          )
        df_all_data_weather_daily.rename(columns={'item_0': item}, inplace=True)

In [40]:
#Выделяем данные о дате и времени в отдельные колонки в датафрейме с почасовыми данными о погоде

df_all_data_weather_hourly[['date', 'hour']]=df_all_data_weather_hourly['time'].str.split('T', expand=True)

In [41]:
#Оставим только необходимые для анализа колонки в датафрейме с почасовыми данными о погоде

df_all_data_weather_hourly=df_all_data_weather_hourly[['latitude', 'longitude',
                    'timezone_abbreviation', 'date', 'hour','weather_code', 
                    'temperature_2m', 'dew_point_2m', 'precipitation', 'rain', 
                    'snowfall', 'snow_depth', 'sunshine_duration', 'wind_speed_10m', 
                    'wind_direction_10m', 'wind_gusts_10m', 
                     'cloud_cover']]

In [42]:
#Выделяем данные о дате и времени восхода солнца в отдельные колонки в датафрейме с дневными данными о погоде

df_all_data_weather_daily[['sunrise_date', 'sunrise_hour']]=df_all_data_weather_daily['sunrise'].str.split('T', expand=True)

In [43]:
#Выделяем данные о дате и времени захода солнца в отдельные колонки в датафрейме с дневными данными о погоде

df_all_data_weather_daily[['sunset_date', 'sunset_hour']]=df_all_data_weather_daily['sunset'].str.split('T', expand=True)

In [44]:
#Изменяем название колонки в датафрейме с дневными данными о погоде

df_all_data_weather_daily.rename(columns={'time': 'date'}, inplace=True)

In [45]:
#Оставим только необходимые для анализа колонки в датафрейме с дневными данными о погоде

df_all_data_weather_daily=df_all_data_weather_daily[['latitude', 'longitude', 'date',  
                    'sunrise_hour', 'sunset_hour', 
                    'daylight_duration', 'precipitation_sum', 
                    'rain_sum', 'snowfall_sum'
                   ]]

In [46]:
#Объединим датафреймы с почасовыми и дневными данными о погоде

df_all_data_weather_hourly=df_all_data_weather_hourly.merge(df_all_data_weather_daily, on=['latitude', 'longitude', 'date'], how='left') 

In [47]:
#Изменяем тип данных полей с координатами на float

df_all_data_weather_hourly['latitude'] = df_all_data_weather_hourly['latitude'].astype('float')
df_all_data_weather_hourly['longitude'] = df_all_data_weather_hourly['longitude'].astype('float')

In [48]:
#Изменяем тип данных с датой

df_all_data_weather_hourly['date'] = pd.to_datetime(df_all_data_weather_hourly['date'])

In [49]:
#Выводим информацию об объединенном датафрейме

df_all_data_weather_hourly.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 175344 entries, 0 to 175343
Data columns (total 23 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   latitude               175344 non-null  float64       
 1   longitude              175344 non-null  float64       
 2   timezone_abbreviation  175344 non-null  object        
 3   date                   175344 non-null  datetime64[ns]
 4   hour                   175344 non-null  object        
 5   weather_code           175344 non-null  int64         
 6   temperature_2m         175344 non-null  float64       
 7   dew_point_2m           175344 non-null  float64       
 8   precipitation          175344 non-null  float64       
 9   rain                   175344 non-null  float64       
 10  snowfall               175344 non-null  float64       
 11  snow_depth             175344 non-null  float64       
 12  sunshine_duration      175344 non-null  floa

In [50]:
#Выводим первые 5 строк объединенного датафрейма

df_all_data_weather_hourly.head()

,latitude,longitude,timezone_abbreviation,date,hour,weather_code,temperature_2m,dew_point_2m,precipitation,rain,...,wind_speed_10m,wind_direction_10m,wind_gusts_10m,cloud_cover,sunrise_hour,sunset_hour,daylight_duration,precipitation_sum,rain_sum,snowfall_sum
0,55.782074,49.124237,GMT+3,2016-01-01,00:00,3,-16.2,-19.1,0.0,0.0,...,15.5,54,28.1,98,08:13,15:20,25612.66,0.0,0.0,0.0
1,55.782074,49.124237,GMT+3,2016-01-01,01:00,3,-16.6,-19.6,0.0,0.0,...,14.7,62,27.7,99,08:13,15:20,25612.66,0.0,0.0,0.0
2,55.782074,49.124237,GMT+3,2016-01-01,02:00,3,-17.0,-20.0,0.0,0.0,...,14.7,62,27.0,84,08:13,15:20,25612.66,0.0,0.0,0.0
3,55.782074,49.124237,GMT+3,2016-01-01,03:00,2,-18.0,-21.0,0.0,0.0,...,15.0,60,27.4,64,08:13,15:20,25612.66,0.0,0.0,0.0
4,55.782074,49.124237,GMT+3,2016-01-01,04:00,1,-18.9,-21.9,0.0,0.0,...,15.2,59,28.4,36,08:13,15:20,25612.66,0.0,0.0,0.0


In [51]:
#Добавим колонку с названием города

df_all_data_weather_hourly['n_p'] = np.where(df_all_data_weather_hourly['longitude']==49.124237, 'Казань', 'Новосибирск')

In [52]:
# Проверяем уникальные значения в категориальных столбцах

for column in ['latitude', 'longitude', 'timezone_abbreviation', 'weather_code', 'n_p']:
    print(f'Уникальные значения в столбце {column}:')
    print(df_all_data_weather_hourly[column].sort_values().unique())
    print()

Уникальные значения в столбце latitude:
[55.079086 55.782074]

Уникальные значения в столбце longitude:
[49.124237 82.99402 ]

Уникальные значения в столбце timezone_abbreviation:
['GMT+3' 'GMT+7']

Уникальные значения в столбце weather_code:
[ 0  1  2  3 51 53 55 61 63 65 71 73 75]

Уникальные значения в столбце n_p:
['Казань' 'Новосибирск']



In [53]:
# Проверяем полные дубликаты в датафрейме

df_all_data_weather_hourly.duplicated().sum()

0

In [ ]:
#Выгружаем обработанные данные о погоде в таблицу базы данных
logger.info("Выгружаем обработанные данные о погоде в таблицу базы данных")
df_all_data_weather_hourly.to_sql('weather', con=engine, if_exists='append', index=False)  
logger.info("Данные о погоде успешно выгружены в таблицу базы данных")

___

## 6. Получение ОКАТО-кодов регионов РФ<a id='soderjanie_6'></a>

In [54]:
#Функция для получения ОКАТО-коды всех регионов РФ (877 - код РФ)
#дата всегда прошлый месяц

'''
Функция для получения ОКАТО-кодов регионов РФ с сайта gibdd.ru.
'''

def get_all_regions(user_agent: str):
    
    #По умолчанию берем самые свежие данные, за месяц перед текущим
    now = datetime.now()
    year = now.year
    month = now.month - 1 if now.month > 1 else 12 
    if month == 12:
        year -= 1
        
    #Получаем список регионов РФ
    logger.info("Получаем список регионов РФ.")  
    rf_payload = {
        "maptype": 1,
        "region": "877", 
        "date": f'["MONTHS:{month}.{year}"]',
        "pok": "1"
    }
    
    #Указываем заголовок
    headers = {
        'User-Agent': user_agent,
        'Content-Type': 'application/json'
    }
    #Отправляем запрос к stat.gibdd.ru
    try:
        response = requests.post(
            "http://stat.gibdd.ru/map/getMainMapData",
            json=rf_payload,
            headers=headers,
            timeout=30
        )
        
        # Проверка ответа
        if response.status_code != 200:
            logger.error(f"Ошибка: {response.status_code}")
            return 
                
        
        result = response.json()
        metabase = json.loads(result["metabase"])
        maps_data = json.loads(metabase[0]["maps"])
        
        #Получаем регионы + ОКТМО
        regions = []
        for region in maps_data:
            regions.append({
                "id": region["id"],
                "name": region["name"],
                "districts": []
            })
        
        logger.info(f"Найдено {len(regions)} регионов")
              
        for i, region in enumerate(regions, 1):
            print(f"[{i}/{len(regions)}] {region['name']} ({region['id']})...")
            
            # Получаем ОКАТО-коды муниципальных образований для всех регионов
            region_payload = {
                "maptype": 1,
                "region": region["id"],
                "date": f'["MONTHS:{month}.{year}"]',
                "pok": "1"
            }
            
            #Отправляем запрос к stat.gibdd.ru
            try:
                reg_response = requests.post(
                    "http://stat.gibdd.ru/map/getMainMapData",
                    json=region_payload,
                    headers=headers,
                    timeout=30
                )
                
                # Проверка ответа
                if reg_response.status_code == 200:
                    reg_result = reg_response.json()
                    reg_metabase = json.loads(reg_result["metabase"])
                    reg_maps_data = json.loads(reg_metabase[0]["maps"])
                    
                    municipalities = []
                    for municipality in reg_maps_data:
                        municipalities.append({
                            "id": municipality["id"],
                            "name": municipality["name"]
                        })
                    
                    region["districts"] = municipalities
                    logger.info(f"найдено {len(municipalities)} муниципалитетов")
                else:
                    logger.error(f"ошибка {reg_response.status_code}")
                    
            except Exception as e:
                logger.error(f"ошибка: {e}")
            
            if i < len(regions):
                time.sleep(0.5)
        
        filename = "regions_all.json"
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(regions, f, ensure_ascii=False, indent=2)
        
        total_municipalities = sum(len(r["districts"]) for r in regions)
        logger.info(f"Файл: {filename}")
        logger.info(f"Регионов: {len(regions)}")
        logger.info(f"Всего муниципалитетов: {total_municipalities}")
                
    except Exception as e:
        logger.error(f"Критическая ошибка: {e}")
        
    return regions

In [55]:
#Выполняем функцию для получения ОКАТО-кодов всех регионов РФ и их сохранения

if __name__ == "__main__": 

    result=get_all_regions(user_agent)

INFO: Получаем список регионов РФ.
INFO: Найдено 90 регионов
INFO: найдено 13 муниципалитетов


[1/90] Ямало-Ненецкий АО (71140)...


INFO: найдено 21 муниципалитетов


[2/90] Ханты-Мансийский АО (71100)...


INFO: найдено 24 муниципалитетов


...

INFO: найдено 23 муниципалитетов


[89/90] Запорожская область (956)...


INFO: найдено 22 муниципалитетов
INFO: Файл: regions_all.json


[90/90] Херсонская область (957)...


INFO: Регионов: 90
INFO: Всего муниципалитетов: 2530


___

## 7. Предобработка данных о кодах ОКТМО и их выгрузка в таблицу базы данных<a id='soderjanie_7'></a>


In [56]:
#Сохраняем полученные ОКАТО-коды в датафрейм

df_regions_code_bufer=pd.json_normalize(result)

In [57]:
#Выводим первые 5 строк датафрейма с ОКАТО-кодами

df_regions_code_bufer.head()

,id,name,districts
0,71140,Ямало-Ненецкий АО,"[{'id': '71166', 'name': 'Шурышкарский район'}..."
1,71100,Ханты-Мансийский АО,"[{'id': '71126', 'name': 'Сургутский район'}, ..."
2,71,Тюменская область,"[{'id': '712581', 'name': 'Ярковский район'}, ..."
3,10011,Ненецкий АО,"[{'id': '11100', 'name': 'Ненецкий АО'}]"
4,99,Еврейская автономная область,"[{'id': '99205', 'name': 'Биробиджанский район..."


In [58]:
# Преобразуем столбец датафрейма с данными о регионах в JSON-строку

for i, item  in enumerate(df_regions_code_bufer['districts']):
    df_regions_code_bufer['districts'][i]=json.dumps(item)

In [ ]:
#Выгружаем полученные данные о кодах ОКАТО в буферную таблицу базы данных

logger.info("Выгружаем полученные данные о кодах ОКАТО в буферную таблицу базы данных")
df_regions_code_bufer.to_sql('bufer_codes_oktmo', con=engine, if_exists='replace', index=False, chunksize=500, method='multi')
logger.info("Данные о кодах ОКАТО успешно выгружены в буферную таблицу базы данных")   

In [59]:
#Сохраняем полученные ОКАТО-коды в датафрейм, нормалтзуя данные с помощью json_normalize

df_regions_code=pd.json_normalize(result,meta=['id', 'name'],
        record_path=['districts'],
        record_prefix='districts_')

In [60]:
#Выводим первые 5 строк датафрейма с ОКАТО-кодами

df_regions_code.head()

,districts_id,districts_name,id,name
0,71166,Шурышкарский район,71140,Ямало-Ненецкий АО
1,71153,Красноселькупский район,71140,Ямало-Ненецкий АО
2,71158,Приуральский район,71140,Ямало-Ненецкий АО
3,71156,Надымский район,71140,Ямало-Ненецкий АО
4,71160,Пуровский район,71140,Ямало-Ненецкий АО


In [61]:
#Выводим информацию о датафрейме

df_regions_code.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2530 entries, 0 to 2529
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   districts_id    2530 non-null   object
 1   districts_name  2530 non-null   object
 2   id              2530 non-null   object
 3   name            2530 non-null   object
dtypes: object(4)
memory usage: 79.2+ KB


In [62]:
# Преобразуем и очистим данные о названии муниципалитета

df_regions_code['districts_name'] = df_regions_code['districts_name'].str.replace('г.', '').str.strip()

/tmp/ipykernel_338/1553382197.py:3: FutureWarning: The default value of regex will change from True to False in a future version.
  df_regions_code['districts_name'] = df_regions_code['districts_name'].str.replace('г.', '').str.strip()


In [63]:
# Проверяем уникальные значения в категориальных столбцах

for column in ['id', 'name']:
    print(f'Уникальные значения в столбце {column}:')
    print(df_regions_code[column].sort_values().unique())
    print(f'Число уникальных значения в столбце {column}:')
    print(df_regions_code[column].nunique())
    print()

Уникальные значения в столбце id:
['1' '10' '10011' '101' '11' '12' '14' '15' '17' '18' '19' '20' '22' '24'
 '25' '26' '27' '28' '29' '3' '30' '32' '33' '34' '35' '36' '37' '38' '4'
 '40' '41' '42' '44' '45' '46' '47' '49' '5' '50' '52' '53' '54' '56' '57'
 '58' '60' '61' '63' '64' '65' '66' '67' '68' '69' '7' '70' '71' '71100'
 '71140' '73' '75' '76' '77' '78' '79' '8' '80' '81' '82' '83' '84' '85'
 '86' '87' '88' '89' '90' '91' '92' '93' '94' '95' '956' '957' '958' '96'
 '960' '97' '98' '99']
Число уникальных значения в столбце id:
90

Уникальные значения в столбце name:
['Алтайский край' 'Амурская область' 'Архангельская область'
 'Астраханская область' 'Белгородская область' 'Брянская область'
 'Владимирская область' 'Волгоградская область' 'Вологодская область'
 'Воронежская область' 'Донецкая Народная Республика'
 'Еврейская автономная область' 'Забайкальский край' 'Запорожская область'
 'Ивановская область' 'Иркутская область'
 'Кабардино-Балкарская Республика' 'Калининградская 

In [ ]:
#Выгружаем полученные данные о кодах ОКТМО в таблицу базы данных

logger.info("Выгружаем полученные данные о кодах ОКТМО в таблицу базы данных")
df_regions_code.to_sql('codes_oktmo', con=engine, if_exists='replace', index=False, chunksize=500, method='multi')
logger.info("Данные о кодах ОКТМО успешно выгружены в таблицу базы данных") 

In [64]:
#Для выбранных городов выводим строки датафрейма с кодами ОКАТО, их содержащие

for city in cities_selected:
    print(df_regions_code[df_regions_code['districts_name'].str.contains(city, case=False, na=False)])

    districts_id districts_name  id                  name
242        92401         Казань  92  Республика Татарстан
     districts_id       districts_name  id                   name
1183        50240  Новосибирский район  50  Новосибирская область
1187        50401          Новосибирск  50  Новосибирская область


___


## 8. Парсинг данных о ДТП через API<a id='soderjanie_8'></a>

In [65]:
#Функция для получения полных карточек ДТП

def get_dtp_cards(region_id, district_id, year, month, user_agent, start=1, end=300):
    
    """
    Получение полных карточек ДТП с пагинацией
    region_id: ОКАТО региона
    district_id: ОКАТО муниципалитета
    year: год 
    month: месяц
    headers
    """
    logger.info(f'Получение карточек ДТП для муниципалитета с кодом ОКАТО {district_id} региона с кодом ОКАТО {region_id} за период {month}.{year}')
    url = "http://stat.gibdd.ru/map/getDTPCardData"
    
    payload = {
        "data": {
            "date": [f"MONTHS:{month}.{year}"],
            "ParReg": region_id,
            "order": {"type": "1", "fieldName": "dat"},
            "reg": district_id,
            "ind": "1",
            "st": str(start),
            "en": str(end),
            "fil": {
                "isSummary": False  # Полные данные вместо сводных
            },
            "fieldNames": [
                "dat", "time", "coordinates", "infoDtp", "k_ul", "dor", "ndu",
                "k_ts", "ts_info", "pdop", "pog", "osv", "s_pch", "s_pog",
                "n_p", "n_pg", "obst", "sdor", "t_osv", "t_p", "t_s", "v_p", "v_v"
            ]
        }
    }

    headers = {
        "User-Agent": user_agent,
        "Accept": "application/json",
        "Content-Type": "application/json"
    }

    try:
        # Двойное кодирование JSON требуется API ГИБДД
        request_data = {
            "data": json.dumps(payload["data"], separators=(',', ':'))
        }

        response = requests.post(
            url,
            json=request_data,
            headers=headers,
            timeout=30
        )

        if response.status_code == 200:
            response_data = json.loads(response.text)
            return json.loads(response_data["data"]).get("tab", [])
        else:
            logger.error(f"Ошибка HTTP: {response.status_code}")
            return None

    except Exception as e:
        logger.error(f"Ошибка при запросе данных: {str(e)}")
        return None

In [66]:
#Выполнение функции для получения полных карточек ДТП

#Сохраняем пустой список для добавления в него карточек ДТП
dtp_data=[]
#Сохраняем спиок с данными ОКАТО выбранных регионов и муниципалитетов
cities_codes_regions=[['92', '92401'],['50','50401']]
#Сохраняем год начала и окончания периода
year_start=2016
year_end=2025

#Сохраняем месяц начала и окончания периода (если данные за полные года скачиваем, то заполняем 1 и 12 соответсвенно)
month_start=1
month_end=12

#Для кадого месяца в рассматриваемом периоде скачиваем карточки ДТП по выбранным регионам 
#и добавляем их к общему списку
for region_id, district_id in cities_codes_regions:
    for year in range(year_start, year_end+1, 1):
        for month in range (month_start, month_end+1, 1):
            dtp_data_m = get_dtp_cards(region_id, district_id, year, month, user_agent)
            dtp_data.append(dtp_data_m)

INFO: Получение карточек ДТП для муниципалитета с кодом ОКАТО 92401 региона с кодом ОКАТО 92 за период 1.2016
INFO: Получение карточек ДТП для муниципалитета с кодом ОКАТО 92401 региона с кодом ОКАТО 92 за период 2.2016
INFO: Получение карточек ДТП для муниципалитета с кодом ОКАТО 92401 региона с кодом ОКАТО 92 за период 3.2016
INFO: Получение карточек ДТП для муниципалитета с кодом ОКАТО 92401 региона с кодом ОКАТО 92 за период 4.2016
INFO: Получение карточек ДТП для муниципалитета с кодом ОКАТО 92401 региона с кодом ОКАТО 92 за период 5.2016
INFO: Получение карточек ДТП для муниципалитета с кодом ОКАТО 92401 региона с кодом ОКАТО 92 за период 6.2016
INFO: Получение карточек ДТП для муниципалитета с кодом ОКАТО 92401 региона с кодом ОКАТО 92 за период 7.2016
INFO: Получение карточек ДТП для муниципалитета с кодом ОКАТО 92401 региона с кодом ОКАТО 92 за период 8.2016
INFO: Получение карточек ДТП для муниципалитета с кодом ОКАТО 92401 региона с кодом ОКАТО 92 за период 9.2016
INFO: Полу

___

## 9. Предобработка данных о ДТП и их выгрузка в таблицу базы данных<a id='soderjanie_9'></a>

In [67]:
#Сохраняем полученные данные о ДТП в датафрейм

dtp_data_bufer=pd.DataFrame(dtp_data)

In [68]:
# Преобразуем ячейки датафрейма с данными о ДТП в JSON-строки

for i, col in enumerate(dtp_data_bufer.columns):
    for t, item  in enumerate(dtp_data_bufer[col]):
        dtp_data_bufer[t][col]=json.dumps(item)

In [69]:
# Сохраним список с новыми наименованиями колонок датафрейма

new_col=[]
for i in dtp_data_bufer.columns:
    new_col.append(f'card_{i}')


In [70]:
# Измением наименование колонок датафрейма

dtp_data_bufer=dtp_data_bufer.set_axis(new_col, axis=1)

In [71]:
#Выводим первые 2 строки датафрейма

dtp_data_bufer.head(2)

,card_0,card_1,card_2,card_3,card_4,card_5,card_6,card_7,card_8,card_9,...,card_230,card_231,card_232,card_233,card_234,card_235,card_236,card_237,card_238,card_239
0,"{""KartId"": 191002725, ""rowNum"": 1, ""date"": ""30...","{""KartId"": 191250381, ""rowNum"": 1, ""date"": ""29...","{""KartId"": 191663975, ""rowNum"": 1, ""date"": ""31...","{""KartId"": 193074994, ""rowNum"": 1, ""date"": ""30...","{""KartId"": 193698958, ""rowNum"": 1, ""date"": ""31...","{""KartId"": 194540075, ""rowNum"": 1, ""date"": ""30...","{""KartId"": 195946925, ""rowNum"": 1, ""date"": ""31...","{""KartId"": 198294092, ""rowNum"": 1, ""date"": ""31...","{""KartId"": 200064030, ""rowNum"": 1, ""date"": ""30...","{""KartId"": 201563475, ""rowNum"": 1, ""date"": ""31...",...,"{""KartId"": 224676485, ""rowNum"": 1, ""date"": ""31...","{""KartId"": 224730238, ""rowNum"": 1, ""date"": ""30...","{""KartId"": 224793246, ""rowNum"": 1, ""date"": ""31...","{""KartId"": 224859253, ""rowNum"": 1, ""date"": ""30...","{""KartId"": 224933769, ""rowNum"": 1, ""date"": ""31...","{""KartId"": 225011577, ""rowNum"": 1, ""date"": ""31...","{""KartId"": 225083416, ""rowNum"": 1, ""date"": ""30...","{""KartId"": 225155738, ""rowNum"": 1, ""date"": ""31...","{""KartId"": 225217286, ""rowNum"": 1, ""date"": ""30...","{""KartId"": 225406749, ""rowNum"": 1, ""date"": ""31..."
1,"""{\""KartId\"": 191250381, \""rowNum\"": 1, \""date...","{""KartId"": 191292286, ""rowNum"": 2, ""date"": ""29...","{""KartId"": 191661939, ""rowNum"": 2, ""date"": ""31...","{""KartId"": 193147932, ""rowNum"": 2, ""date"": ""30...","{""KartId"": 193729959, ""rowNum"": 2, ""date"": ""31...","{""KartId"": 194540076, ""rowNum"": 2, ""date"": ""30...","{""KartId"": 195946926, ""rowNum"": 2, ""date"": ""31...","{""KartId"": 198293923, ""rowNum"": 2, ""date"": ""31...","{""KartId"": 199960844, ""rowNum"": 2, ""date"": ""30...","{""KartId"": 201545783, ""rowNum"": 2, ""date"": ""31...",...,"{""KartId"": 224676614, ""rowNum"": 2, ""date"": ""31...","{""KartId"": 224746916, ""rowNum"": 2, ""date"": ""30...","{""KartId"": 224793663, ""rowNum"": 2, ""date"": ""31...","{""KartId"": 224986639, ""rowNum"": 2, ""date"": ""30...","{""KartId"": 224933736, ""rowNum"": 2, ""date"": ""31...","{""KartId"": 225013274, ""rowNum"": 2, ""date"": ""30...","{""KartId"": 225084769, ""rowNum"": 2, ""date"": ""30...","{""KartId"": 225155799, ""rowNum"": 2, ""date"": ""31...","{""KartId"": 225217333, ""rowNum"": 2, ""date"": ""30...","{""KartId"": 225349539, ""rowNum"": 2, ""date"": ""31..."


In [ ]:
#Выгружаем полученные данные о ДТП в буферную таблицу базы данных

logger.info("Выгружаем полученные данные о ДТП в буферную таблицу базы данных")
for i, col in enumerate(dtp_data_bufer.columns):
    if i==0:
        df_col = dtp_data_bufer[col]
        df_col.to_sql('bufer_dtp', con=engine, if_exists='replace', chunksize=500, method='multi')

    else:
        df_col= dtp_data_bufer[col]
        df_col.to_sql('temp_table', con=engine, if_exists='replace',  chunksize=500, method='multi')
        with engine.connect() as conn:
            conn.execute(f"ALTER TABLE bufer_dtp ADD COLUMN {col} text")
            conn.execute(f"UPDATE bufer_dtp SET {col}=(SELECT {col} FROM temp_table  WHERE bufer_dtp.index = temp_table.index)")
            conn.execute("DROP TABLE temp_table")
             
logger.info("Данные о ДТП успешно выгружены в буферную таблицу базы данных") 

In [72]:
#Основной DataFrame с данными о ДТП (только верхний уровень)

main_data = []
for i in range(len(dtp_data)):
    for item in dtp_data[i]:
        main_item = {
        'KartId': item['KartId'],
        'rowNum': item['rowNum'],
        'date': item['date'],
        'Time': item['Time'],
        'District': item['District'],
        'DTP_V': item['DTP_V'],
        'POG': item['POG'],
        'RAN': item['RAN'],
        'K_TS': item['K_TS'],
        'K_UCH': item['K_UCH'],
        'emtp_number': item['emtp_number']
    }
        main_data.append(main_item)

main_df = pd.DataFrame(main_data)

In [73]:
#Выводим информацию о датафрейме

main_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30608 entries, 0 to 30607
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   KartId       30608 non-null  int64 
 1   rowNum       30608 non-null  int64 
 2   date         30608 non-null  object
 3   Time         30608 non-null  object
 4   District     30608 non-null  object
 5   DTP_V        30608 non-null  object
 6   POG          30608 non-null  int64 
 7   RAN          30608 non-null  int64 
 8   K_TS         30608 non-null  int64 
 9   K_UCH        30608 non-null  int64 
 10  emtp_number  30608 non-null  object
dtypes: int64(6), object(5)
memory usage: 2.6+ MB


In [74]:
#Выводим первые 5 строк датафрейма

main_df.head()

,KartId,rowNum,date,Time,District,DTP_V,POG,RAN,K_TS,K_UCH,emtp_number
0,191002725,1,30.01.2016,16:51,Московский р-н,Наезд на пешехода,0,1,1,2,
1,190997161,2,30.01.2016,11:05,Новосавиновский р-н,Наезд на пешехода,0,1,1,2,
2,190998969,3,30.01.2016,01:10,Авиастроительный р-н,Столкновение,0,1,2,3,
3,190996837,4,30.01.2016,09:10,Приволжский р-н,Столкновение,0,1,3,3,
4,190991145,5,30.01.2016,10:15,Вахитовский р-н,Столкновение,0,1,2,3,


In [75]:
#Приводим заголовки столбцов датафрейма в нижний регистр

main_df.columns = main_df.columns.str.lower()

In [76]:
# Изменим тип данных в столбце с датой

main_df['date'] = pd.to_datetime(main_df['date'])

In [77]:
# Добавим столбец со временем, округленным до часа

main_df['time'] = pd.to_datetime(main_df['time'])
main_df['hour'] = main_df['time'].dt.strftime('%H:00')
main_df['time'] =main_df['time'].dt.strftime('%H:%M')

In [78]:
# Проверяем количество уникальных значений в столбце с номерами карточек ДТП

main_df['kartid'].nunique()

30608

In [79]:
# Проверяем уникальные значения в категориальных столбцах

for column in ['dtp_v', 'pog', 'ran', 'k_ts', 'k_uch', 'emtp_number']:
    print(f'Уникальные значения в столбце {column}:')
    print(main_df[column].sort_values().unique())
    print(f'Число уникальных значения в столбце {column}:')
    print(main_df[column].nunique())
    print()

Уникальные значения в столбце dtp_v:
['Возгорание вследствие технической неисправности движущегося или остановившегося ТС, участвующего в дорожном движении.'
 'Иной вид ДТП' 'Наезд на велосипедиста'
 'Наезд на внезапно возникшее препятствие' 'Наезд на животное'
 'Наезд на лицо, использующее для передвижения СИМ'
 'Наезд на лицо, не являющееся участником дорожного движения, осуществляющее какую-либо другую деятельность'
 'Наезд на лицо, не являющееся участником дорожного движения, осуществляющее несение службы'
 'Наезд на лицо, не являющееся участником дорожного движения, осуществляющее производство работ'
 'Наезд на пешехода' 'Наезд на препятствие' 'Наезд на стоящее ТС'
 'Опрокидывание' 'Отбрасывание предмета' 'Падение груза'
 'Падение пассажира' 'Столкновение' 'Съезд с дороги']
Число уникальных значения в столбце dtp_v:
18

Уникальные значения в столбце pog:
[0 1 2 3 4]
Число уникальных значения в столбце pog:
5

Уникальные значения в столбце ran:
[ 0  1  2  3  4  5  6  7  8  9 10 11 

In [80]:
# Определяем количество явных дубликатов

main_df.duplicated().sum()

0

In [ ]:
#Выгружаем полученные общие данные о ДТП в таблицу базы данных

logger.info("Выгружаем полученные общие данные о ДТП в таблицу базы данных")
main_df.to_sql('dtp_common', con=engine, if_exists='append', index=False, chunksize=500, method='multi')
logger.info("Общие данные о ДТП успешно выгружены в таблицу базы данных")

___

In [81]:
#DataFrame с общей информацией о ДТП (infoDtp)

info_dtp_list = []
for i in range(len(dtp_data)):
    for item in dtp_data[i]:
        info_row = {
        'KartId': item['KartId'],
        'n_p': item['infoDtp'].get('n_p', ''),
        'street': item['infoDtp'].get('street', ''),
        'house': item['infoDtp'].get('house', ''),
        'dor': item['infoDtp'].get('dor', ''),
        'km': item['infoDtp'].get('km', ''),
        'm': item['infoDtp'].get('m', ''),
        'k_ul': item['infoDtp'].get('k_ul', ''),
        'dor_k': item['infoDtp'].get('dor_k', ''),
        'dor_z': item['infoDtp'].get('dor_z', ''),
        's_pch': item['infoDtp'].get('s_pch', ''),
        'osv': item['infoDtp'].get('osv', ''),
        'change_org_motion': item['infoDtp'].get('change_org_motion', ''),
        's_dtp': item['infoDtp'].get('s_dtp', ''),
        'COORD_W': item['infoDtp'].get('COORD_W', ''),
        'COORD_L': item['infoDtp'].get('COORD_L', ''),
        # Преобразуем списки в строки
        'ndu': ', '.join(item['infoDtp'].get('ndu', [])) if item['infoDtp'].get('ndu') else '',
        'sdor': ', '.join(item['infoDtp'].get('sdor', [])) if item['infoDtp'].get('sdor') else '',
        'factor': ', '.join(item['infoDtp'].get('factor', [])) if item['infoDtp'].get('factor') else '',
        's_pog': ', '.join(item['infoDtp'].get('s_pog', [])) if item['infoDtp'].get('s_pog') else '',
        'OBJ_DTP': ', '.join(item['infoDtp'].get('OBJ_DTP', [])) if item['infoDtp'].get('OBJ_DTP') else ''
    }
        info_dtp_list.append(info_row)

info_dtp_df = pd.DataFrame(info_dtp_list)

In [82]:
#Выводим информацию о датафрейме 

info_dtp_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30608 entries, 0 to 30607
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   KartId             30608 non-null  int64 
 1   n_p                30608 non-null  object
 2   street             30608 non-null  object
 3   house              30608 non-null  object
 4   dor                30608 non-null  object
 5   km                 30608 non-null  object
 6   m                  30608 non-null  object
 7   k_ul               30608 non-null  object
 8   dor_k              30608 non-null  object
 9   dor_z              30608 non-null  object
 10  s_pch              30608 non-null  object
 11  osv                30608 non-null  object
 12  change_org_motion  30608 non-null  object
 13  s_dtp              30608 non-null  object
 14  COORD_W            30608 non-null  object
 15  COORD_L            30608 non-null  object
 16  ndu                30608 non-null  objec

In [83]:
#Выводим первые 5 строк датафрейма

info_dtp_df.head()

,KartId,n_p,street,house,dor,km,m,k_ul,dor_k,dor_z,...,osv,change_org_motion,s_dtp,COORD_W,COORD_L,ndu,sdor,factor,s_pog,OBJ_DTP
0,191002725,г Казань,пр-кт Ибрагимова,14,,0,0,Магистральные улицы общегородского значения,,Не указано,...,"В темное время суток, освещение включено",Режим движения не изменялся,820,55.838855,49.087961,Не установлены,Перегон (нет объектов на месте ДТП),Сведения отсутствуют,"Пасмурно, Снегопад",Остановка общественного транспорта
1,190997161,г Казань,ул Чистопольская,33 В,,0,0,Магистральные дороги,,Не указано,...,Светлое время суток,Режим движения не изменялся,820,55.819364,49.122089,Недостатки зимнего содержания,Перегон (нет объектов на месте ДТП),Сведения отсутствуют,Пасмурно,Многоквартирные жилые дома
2,190998969,г Казань,ул Беломорская,Челюскина,,0,0,Магистральные улицы общегородского значения,,Не указано,...,"В темное время суток, освещение включено",Режим движения не изменялся,300,55.865159,49.078945,Недостатки зимнего содержания,"Перегон (нет объектов на месте ДТП), Регулируе...",Сведения отсутствуют,Снегопад,"Регулируемый перекрёсток, Многоквартирные жилы..."
3,190996837,г Казань,ул Рихарда Зорге,82,,0,0,Магистральные улицы общегородского значения,,Не указано,...,Светлое время суток,Режим движения не изменялся,070,55.745793,49.21574,Не установлены,Перегон (нет объектов на месте ДТП),Сведения отсутствуют,Пасмурно,Регулируемый перекрёсток
4,190991145,г Казань,ул Горького,Жуковского,,0,0,Магистральные дороги,,Не указано,...,Светлое время суток,Режим движения не изменялся,420,55.794047,49.129092,Недостатки зимнего содержания,"Перегон (нет объектов на месте ДТП), Регулируе...",Отключение электроснабжения на данном элементе...,Пасмурно,Остановка общественного транспорта


In [84]:
#Приводим заголовки столбцов датафрейма в нижний регистр

info_dtp_df.columns = info_dtp_df.columns.str.lower()

In [85]:
# Проверяем уникальные значения в категориальных столбцах

for column in ['n_p', 'km', 'm', 'k_ul', 'dor_k', 'dor_z', 's_pch', 'osv', 'change_org_motion', 's_dtp',  'ndu', 'sdor', 'factor', 's_pog', 'obj_dtp', 'coord_w', 'coord_l']:
    print(f'Уникальные значения в столбце {column}:')
    print(info_dtp_df[column].sort_values().unique())
    print(f'Число уникальных значения в столбце {column}:')
    print(info_dtp_df[column].nunique())
    print()

Уникальные значения в столбце n_p:
['г Казань' 'г Новосибирск' 'д Самосырово' 'с Константиновка' 'с Чебакса']
Число уникальных значения в столбце n_p:
5

Уникальные значения в столбце km:
['0' '1' '10' '11' '12' '126' '13' '14' '15' '16' '17' '18' '19' '2' '20'
 '29' '3' '30' '4' '5' '6' '7' '777' '778' '779' '780' '781' '782' '783'
 '784' '785' '786' '787' '788' '795' '796' '797' '798' '8' '802' '808' '9'
 '967']
Число уникальных значения в столбце km:
43

Уникальные значения в столбце m:
['0' '10' '100' '130' '137' '150' '158' '165' '200' '210' '217' '225'
 '240' '255' '280' '30' '300' '337' '350' '400' '420' '450' '460' '470'
 '480' '500' '53' '55' '550' '557' '570' '600' '617' '630' '650' '671'
 '700' '710' '736' '738' '750' '800' '810' '820' '834' '840' '845' '875'
 '885' '900' '910' '940' '950' '960' '988']
Число уникальных значения в столбце m:
55

Уникальные значения в столбце k_ul:
['' 'Велосипедные дорожки' 'Иные места' 'Магистральные дороги'
 'Магистральные улицы общегородск

In [86]:
info_dtp_df['n_p'] = info_dtp_df['n_p'].str.replace('г ', '').str.strip()

In [87]:
# Определяем количество явных дубликатов

info_dtp_df.duplicated().sum()

0

In [88]:
# Проверяем количество уникальных значений в столбце с номерами карточек ДТП

info_dtp_df['kartid'].nunique()

30608

In [ ]:
#Выгружаем полученные расширенные данные о ДТП в таблицу базы данных

logger.info("Выгружаем полученные расширенные данные о месте и условиях ДТП в таблицу базы данных")
info_dtp_df.to_sql('dtp_info', con=engine, if_exists='append', index=False, chunksize=500, method='multi')
logger.info("Расширенные данные о месте и условиях данные о ДТП успешно выгружены в таблицу базы данных")

___

In [89]:
#DataFrame с информацией о транспортных средствах

ts_data = []
for i in range(len(dtp_data)):
    for item in dtp_data[i]:
        kart_id = item['KartId']
        if 'ts_info' in item['infoDtp']:
            for ts in item['infoDtp']['ts_info']:
                ts_row = {
                'KartId': kart_id,
                'n_ts': ts.get('n_ts', ''),
                'ts_s': ts.get('ts_s', ''),
                't_ts': ts.get('t_ts', ''),
                'marka_ts': ts.get('marka_ts', ''),
                'm_ts': ts.get('m_ts', ''),
                'color': ts.get('color', ''),
                'r_rul': ts.get('r_rul', ''),
                'g_v': ts.get('g_v', ''),
              
                    'm_pov': ts.get('m_pov', ''),
                't_n': ts.get('t_n', ''),
                'f_sob': ts.get('f_sob', ''),
                'o_pf': ts.get('o_pf', '')
            }
                ts_data.append(ts_row)

ts_df = pd.DataFrame(ts_data)

In [90]:
#Выводим информацию о датафрейме

ts_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49937 entries, 0 to 49936
Data columns (total 13 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   KartId    49937 non-null  int64 
 1   n_ts      49937 non-null  object
 2   ts_s      49937 non-null  object
 3   t_ts      49937 non-null  object
 4   marka_ts  49937 non-null  object
 5   m_ts      49937 non-null  object
 6   color     49937 non-null  object
 7   r_rul     49937 non-null  object
 8   g_v       49937 non-null  object
 9   m_pov     49937 non-null  object
 10  t_n       49937 non-null  object
 11  f_sob     49937 non-null  object
 12  o_pf      49937 non-null  object
dtypes: int64(1), object(12)
memory usage: 5.0+ MB


In [91]:
#Выводим первые 5 строк датафрейма

ts_df.head()

,KartId,n_ts,ts_s,t_ts,marka_ts,m_ts,color,r_rul,g_v,m_pov,t_n,f_sob,o_pf
0,191002725,1,Нет,"В-класс (малый) до 3,9 м",ВАЗ,Жигули ВАЗ-2107 модификации,Иные цвета,Задний,2006,Передний левый угол,Технические неисправности отсутствуют,Частная собственность,Физические лица
1,190997161,1,Нет,"В-класс (малый) до 3,9 м",SKODA,Fabia,Белый,Передний,2007,Передний левый угол,Технические неисправности отсутствуют,Частная собственность,Физические лица
2,190998969,1,Нет,"С-класс (малый средний, компактный) до 4,3 м",KIA,Rio,Серый,Передний,2014,"Передний правый бок (передняя дверь пассажира,...",Технические неисправности отсутствуют,Частная собственность,Физические лица
3,190998969,2,Нет,"С-класс (малый средний, компактный) до 4,3 м",ВАЗ,Kalina,Красный,Передний,2007,Передний левый угол,Технические неисправности отсутствуют,Частная собственность,Физические лица
4,190996837,1,Нет,"В-класс (малый) до 3,9 м",AUDI,Q7,Белый,Задний,2013,Передний левый угол,Технические неисправности отсутствуют,Частная собственность,Физические лица


In [92]:
#Приводим заголовки столбцов датафрейма в нижний регистр

ts_df.columns = ts_df.columns.str.lower()

In [93]:
# Проверяем уникальные значения в категориальных столбцах

for column in ['n_ts', 'ts_s', 't_ts', 'marka_ts', 'm_ts', 'color', 'r_rul', 't_n', 'f_sob', 'o_pf']:
    print(f'Уникальные значения в столбце {column}:')
    print(ts_df[column].sort_values().unique())
    print(f'Число уникальных значения в столбце {column}:')
    print(ts_df[column].nunique())
    print()

Уникальные значения в столбце n_ts:
['1' '2' '3' '4' '5' '6' '7' '8' '9']
Число уникальных значения в столбце n_ts:
9

Уникальные значения в столбце ts_s:
['Да' 'Нет' 'Осталось на месте ДТП'
 'Скрылось и установлено в срок до 1 суток'
 'Скрылось и установлено в срок от 1 до 3 суток'
 'Скрылось и установлено в срок от 3 до 10 суток'
 'Скрылось и установлено в срок свыше 10 суток' 'Скрылось с места ДТП'
 'до 1 сут.' 'от 1  до 3 сут.' 'от 3 до 10 сут.' 'свыше 10 сут.']
Число уникальных значения в столбце ts_s:
12

Уникальные значения в столбце t_ts:
['' 'D-класс (средний) до 4,6 м'
 'S-класс (высший, представительский класс) более 4,9 м'
 'А-класс (особо малый) до 3,5 м' 'Автобетоносмесители'
 'Автобусы (без типа)' 'Автогрейдеры'
 'Автокраны и транспортные средства, оснащенные кранами-манипуляторами'
 'Автомобили скорой медицинской помощи' 'Автоцементовозы' 'Автоэвакуаторы'
 'Бортовые' 'Бортовые грузовые автомобили' 'В-класс (малый) до 3,9 м'
 'Велосипед с двигателем внутреннего сгорания'

In [94]:
# Определяем количество явных дубликатов

ts_df.duplicated().sum()

0

In [95]:
# Проверяем количество уникальных значений в столбце с номерами карточек ДТП

ts_df['kartid'].nunique()

30608

In [ ]:
#Выгружаем полученные данные о транспортных средствах ДТП в таблицу базы данных

logger.info("Выгружаем полученные данные о транспортных средствах ДТП в таблицу базы данных")
ts_df.to_sql('dtp_transp_sr', con=engine, if_exists='append', index=False, chunksize=500, method='multi')
logger.info("Данные о транспортных средствах ДТП успешно выгружены в таблицу базы данных")

____

In [96]:
#DataFrame с участниками ДТП (из ts_uch)

uch_data = []
for i in range(len(dtp_data)):
    for item in dtp_data[i]:
        kart_id = item['KartId']
        if 'ts_info' in item['infoDtp']:
            for ts in item['infoDtp']['ts_info']:
                n_ts = ts.get('n_ts', '')
                if 'ts_uch' in ts and ts['ts_uch']:
                    for uch in ts['ts_uch']:
                        uch_row = {
                        'KartId': kart_id,
                        'n_ts': n_ts,
                        'K_UCH': uch.get('K_UCH', ''),
                        'S_T': uch.get('S_T', ''),
                        'POL': uch.get('POL', ''),
                        'V_ST': uch.get('V_ST', ''),
                        'ALCO': uch.get('ALCO', ''),
                        'SAFETY_BELT': uch.get('SAFETY_BELT', ''),
                        'S_SM': uch.get('S_SM', ''),
                        'N_UCH': uch.get('N_UCH', ''),
                        'S_SEAT_GROUP': uch.get('S_SEAT_GROUP', ''),
                        'INJURED_CARD_ID': uch.get('INJURED_CARD_ID', ''),
                        # Преобразуем списки нарушений в строки
                        'NPDD': ', '.join(uch.get('NPDD', [])) if uch.get('NPDD') else '',
                        'SOP_NPDD': ', '.join(uch.get('SOP_NPDD', [])) if uch.get('SOP_NPDD') else ''
                    }
                        uch_data.append(uch_row)

uch_data_df = pd.DataFrame(uch_data)

In [97]:
#Выводим информацию о датафрейме

uch_data_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61282 entries, 0 to 61281
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   KartId           61282 non-null  int64 
 1   n_ts             61282 non-null  object
 2   K_UCH            61282 non-null  object
 3   S_T              61282 non-null  object
 4   POL              61282 non-null  object
 5   V_ST             61282 non-null  object
 6   ALCO             61282 non-null  object
 7   SAFETY_BELT      61282 non-null  object
 8   S_SM             61282 non-null  object
 9   N_UCH            61282 non-null  object
 10  S_SEAT_GROUP     61282 non-null  object
 11  INJURED_CARD_ID  61282 non-null  object
 12  NPDD             61282 non-null  object
 13  SOP_NPDD         61282 non-null  object
dtypes: int64(1), object(13)
memory usage: 6.5+ MB


In [98]:
#Выводим первые 5 строк датафрейма

uch_data_df.head()

,KartId,n_ts,K_UCH,S_T,POL,V_ST,ALCO,SAFETY_BELT,S_SM,N_UCH,S_SEAT_GROUP,INJURED_CARD_ID,NPDD,SOP_NPDD
0,191002725,1,Водитель,Не пострадал,Мужской,1,,Да,Нет (не скрывался),1,,,Нет нарушений,Нет нарушений
1,190997161,1,Водитель,Не пострадал,Мужской,20,,Да,Нет (не скрывался),2,,,Нет нарушений,Нет нарушений
2,190998969,1,Водитель,Не пострадал,Мужской,5,,Да,Нет (не скрывался),1,,,Несоблюдение очередности проезда,Нет нарушений
3,190998969,1,Пассажир,"Раненый, находящийся (находившийся) на амбула...",Женский,,,Да,Нет (не скрывался),3,,,Нет нарушений,Нет нарушений
4,190998969,2,Водитель,Не пострадал,Мужской,2,,Да,Нет (не скрывался),2,,,Нет нарушений,Нет нарушений


In [99]:
#Приводим заголовки столбцов датафрейма в нижний регистр 

uch_data_df.columns = uch_data_df.columns.str.lower()

In [100]:
# Проверяем уникальные значения в категориальных столбцах

for column in ['n_ts', 'k_uch', 's_t', 'pol', 'v_st', 'alco', 'safety_belt', 's_sm', 'n_uch', 's_seat_group', 'injured_card_id', 'npdd','sop_npdd']:
    print(f'Уникальные значения в столбце {column}:')
    print(uch_data_df[column].sort_values().unique())
    print(f'Число уникальных значения в столбце {column}:')
    print(uch_data_df[column].nunique())
    print()

Уникальные значения в столбце n_ts:
['1' '2' '3' '4' '5' '6' '7' '8' '9']
Число уникальных значения в столбце n_ts:
9

Уникальные значения в столбце k_uch:
['Велосипедист' 'Велосипедист (не применяется)' 'Водитель' 'Пассажир'
 'Пешеход']
Число уникальных значения в столбце k_uch:
5

Уникальные значения в столбце s_t:
['' 'Не пострадал'
 'Получил телесные повреждения с показанием к лечению в медицинских организациях (кроме разовой медицинской помощи)'
 'Получил телесные повреждения с показанием к лечению в медицинских организациях, фактически лечение не проходил, к категории раненый не относится'
 'Получил травмы с оказанием разовой медицинской помощи, к категории раненый не относится'
 'Раненый, находящийся (находившийся)  на амбулаторном лечении, либо которому по характеру полученных травм обозначена необходимость амбулаторного лечения (вне зависимости от его фактического прохождения)'
 'Раненый, находящийся (находившийся) на амбулаторном лечении, либо в условиях дневного стационара'


In [101]:
# Определяем количество явных дубликатов

uch_data_df.duplicated().sum()

0

In [102]:
# Проверяем количество уникальных значений в столбце с номерами карточек ДТП

uch_data_df['kartid'].nunique()

30605

In [ ]:
#Выгружаем полученные данные об участниках ДТП в таблицу базы данных

logger.info("Выгружаем полученные данные об участниках ДТП в таблицу базы данных")
uch_data_df.to_sql('dtp_uchastnic', con=engine, if_exists='append', index=False, chunksize=500, method='multi')
logger.info("Данные об участниках ДТП успешно выгружены в таблицу базы данных")

___

In [103]:
#DataFrame с дополнительными участниками (из uchInfo)

uch_info_data = []
for i in range(len(dtp_data)):
    for item in dtp_data[i]:
        kart_id = item['KartId']
        if 'uchInfo' in item and item['uchInfo']:
            for uch in item['uchInfo']:
                uch_row = {
                'KartId': kart_id,
                'n_ts': 'N/A',  # Для дополнительных участников нет привязки к ТС
                'K_UCH': uch.get('K_UCH', ''),
                'S_T': uch.get('S_T', ''),
                'POL': uch.get('POL', ''),
                'V_ST': uch.get('V_ST', ''),
                'ALCO': uch.get('ALCO', ''),
                'SAFETY_BELT': 'N/A',  # Для дополнительных участников нет этой информации
                'S_SM': uch.get('S_SM', ''),
                'N_UCH': uch.get('N_UCH', ''),
                'S_SEAT_GROUP': 'N/A',
                'INJURED_CARD_ID': 'N/A',
                # Преобразуем списки нарушений в строки
                'NPDD': ', '.join(uch.get('NPDD', [])) if uch.get('NPDD') else '',
                'SOP_NPDD': ', '.join(uch.get('SOP_NPDD', [])) if uch.get('SOP_NPDD') else ''
            }
                uch_info_data.append(uch_row)
                
uch_info_data_df=pd.DataFrame(uch_info_data)

In [104]:
#Выводим первые 5 строк датафрейма

uch_info_data_df.head()

""


_____

## 10. Вывод<a id='soderjanie_10'></a>

В ходе выполнения проекта по выявлению и анализу взаимосвязей между данными о погодных условиях и ДТП в выбранных городах РФ (Казань и Новосибирск) был разработан автоматизированный pipeline, который:
1. Выполняет парсинг данных о всех городах РФ с веб-страницы сайта Wiki с применением инструментов библиотеки Beautiful Soup; 
2. Добавляет координаты к данным о городах (с помощью геокодера Nominatim);
3. Выполняет парсинг данных о погоде через API с сайта open-meteo.com за период с 01.01.2016 по 31.12.2025;
4. Получает коды ОКАТО регионов РФ через API с сайта gibdd.ru;
5. Скачивает карточки ДТП через API с сайта gibdd.ru за период с 01.01.2016 по 31.12.2025;
6. Проводит нормализацию и обработку данных;
7. Загружает полученную информацию в базу данных: в виде промежуточных буферных таблиц с необработанными данными, полученными сразу после скачивания, и финальных таблиц с нормализованными данными.

В результате в базе данных сохранены следующие таблицы с обработанными данными:
- **'cities'**: содержит информацию о городах РФ с их координатами;
- **'codes_oktmo'**: содержит информацию о кодах ОКТМО регионов и муниципалитетов РФ;
- **'weather'**: содержит информацию о погоде в Казани и Новосибирске по часам за период с 01.01.2016 по 31.12.2025; 
- **'dtp_common'**: содержит общую информацию о ДТП в Казани и Новосибирске за период с 01.01.2016 по 31.12.2025;
- **'dtp_info'**: содержит дополнительную информацию о месте и условиях ДТП в Казани и Новосибирске за период с 01.01.2016 по 31.12.2025;
- **'dtp_transp_sr'**: содержит информацию о транспортных средствах, участвующих в ДТП в Казани и Новосибирске за период с 01.01.2016 по 31.12.2025;
- **'dtp_uchastnic'**: содержит информацию об участниках ДТП в Казани и Новосибирске за период с 01.01.2016 по 31.12.2025.

Информации о дополнительных участниках в Казани и Новосибирске за период с 01.01.2016 по 31.12.2025 отсутствует.

